# JN5 - Year & cycle tagging

**Curriculum notebook 5 of 6.** A completion has a date. That one date answers **three different questions** - and the central trap of housing reporting is treating them as one. Conflate them and your RHNA totals are simply wrong. This notebook tags JN4's completions using the **real** `housing_rules` functions, and makes the non-conflation *visible* on a real building.

> Clonable + **read-only**.

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally** (it detects a checkout and skips). On Colab / a bare session it recreates the minimal repo layout under the working directory so the config cell below finds everything unchanged.

In [1]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
_have_repo = (_here/'scripts'/'build_v2').exists() or any((p/'scripts'/'build_v2').exists() for p in _here.parents)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


local repo detected - no fetch needed


## Config

In [2]:
# === CONFIG - point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PERMIT_GLOB = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW  = 7
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root:', REPO_ROOT)


repo root: /Users/johngage/berkeley-data


## Rebuild JN4's completions (recap)

In [3]:
import pandas as pd
from collections import defaultdict, Counter
from housing_predicates import is_housing, net_units
from s0_keys import normalize_address
from cpra_dedup import extract_master_permit

def load(path):
    d = pd.read_excel(path, dtype=str, header=HEADER_ROW); d.columns = [str(c).strip() for c in d.columns]; return d
def pdate(x):
    d = pd.to_datetime(str(x), errors='coerce'); return d.date() if pd.notna(d) else None
df = pd.concat([load(f) for f in glob.glob(PERMIT_GLOB)], ignore_index=True)
df = df[df['PermitNumber'].notna()].rename(columns={'Issuance Date': 'IssuanceDate', 'Finaled Date': 'FinaledDate'}).copy()
df['isnew'] = df['Work Type'].astype(str).str.strip() == 'New'
df = df[[is_housing(o, u, n, a) for o, u, n, a in zip(df['OccType'], df['UnitsAdded'], df['NumberUnits'], df['ADU'])]]
bld = defaultdict(lambda: {'units': 0.0, 'hasnew': False, 'issue': [], 'final': []})
for r in df.itertuples(index=False):
    st = r.StreetType; st = '' if (st is None or str(st).strip().lower() == 'nan') else str(st)
    k = normalize_address(f'{r.StreetNumber} {r.StreetName} {st}'.strip())
    if not k.number: continue
    b = bld[(k.number, k.street, k.stype)]
    b['units'] = max(b['units'], net_units(r.isnew, r.UnitsAdded, r.NumberUnits, r.ADU))
    if r.isnew: b['hasnew'] = True
    pn = str(r.PermitNumber)
    if extract_master_permit(pn) == pn and net_units(r.isnew, r.UnitsAdded, r.NumberUnits, r.ADU) > 0:
        i, f = pdate(r.IssuanceDate), pdate(r.FinaledDate)
        if i: b['issue'].append((i, pn))
        if f: b['final'].append((f, pn))
spine = {k: b for k, b in bld.items() if b['hasnew'] or b['units'] > 0}
completions = {k: b for k, b in spine.items() if b['final']}
print(f'{len(completions)} completions to tag')


951 completions to tag


## Three questions, three functions (import the real `housing_rules`)

The same CO/BP date answers three *different* questions. Don't memorize a list - learn what each one **answers**:

| function | the question it answers | rule |
|---|---|---|
| `cycle_for_date` -> **calendar_cycle** | *Which 8-year cycle does this date sit in?* | 5th / 6th at the **2023-01-31** boundary |
| `rhna_credit_cycle` | *Which RHNA allocation does this unit's credit count toward?* | 6th iff **first-BP >= 2022-06-30** (no upper cap) |
| `is_projection_period` | *Is this date in the narrow bridge window?* | `True` inside **2022-06-30 -> 2023-01-30** only |

And separately, plain calendar arithmetic answers **reporting_year** = *when did it complete?* (the CO date's calendar year - what HCD Table A2 reports by year).

In [4]:
from housing_rules import cycle_for_date, is_projection_period, rhna_credit_cycle
print('imported the real housing_rules classifiers')

imported the real housing_rules classifiers


## reporting_year - *when did it complete?*

The simplest of the three, and the one the **minimal APR (Table A2)** actually needs: the calendar year of the CO date.

In [5]:
def reporting_year(b):
    return max(b['final'])[0].year        # the CO date is MAX-finaled (JN4)
by_year = Counter(reporting_year(b) for b in completions.values())
for y in range(2018, 2026):
    print(f'  {y}: {by_year[y]:>3} completions')
print(f'  total: {sum(by_year.values())}')

  2018:  65 completions
  2019:  98 completions
  2020:  77 completions
  2021: 116 completions
  2022: 101 completions
  2023: 162 completions
  2024: 141 completions
  2025: 191 completions
  total: 951


## The non-conflation, made visible: same date, different cycle answers

Here is the heart of it. Take **1951 Shattuck** (a real 163-unit building). Its **first building permit** was issued **2022-09-08** - inside the projection window. Ask that one date the two cycle questions:

- *Which cycle does the date sit in?* -> `cycle_for_date` = **5th** (it is before 2023-01-31).
- *Which allocation does its credit count toward?* -> `rhna_credit_cycle` = **6th** (it is on/after 2022-06-30).

**The same date gives 5th and 6th** - because they answer different questions. A pipeline that treats "cycle" as one number puts these 163 units in the wrong RHNA total.

In [6]:
k = normalize_address('1951 Shattuck Ave')
b = spine[(k.number, k.street, k.stype)]
first_bp = min(b['issue'])[0]
print(f'1951 Shattuck  ({int(b["units"])} units)  first-BP = {first_bp}')
print(f'  reporting_year (CO year)     : {reporting_year(b)}')
print(f'  calendar_cycle (of first-BP) : {cycle_for_date(first_bp)}   <- which cycle the date sits in')
print(f'  rhna_credit_cycle            : {rhna_credit_cycle(first_bp)}   <- which allocation the credit counts toward')
print(f'  in_projection_period         : {is_projection_period(first_bp)}')
print()
diverge = [k for k, b in completions.items() if b['issue']
           and cycle_for_date(min(b['issue'])[0]) != rhna_credit_cycle(min(b['issue'])[0])]
print(f'completed buildings where calendar_cycle != rhna_credit_cycle: {len(diverge)}  (the projection-credit class)')

1951 Shattuck  (163 units)  first-BP = 2022-09-08
  reporting_year (CO year)     : 2024
  calendar_cycle (of first-BP) : 5th   <- which cycle the date sits in
  rhna_credit_cycle            : 6th   <- which allocation the credit counts toward
  in_projection_period         : True

completed buildings where calendar_cycle != rhna_credit_cycle: 77  (the projection-credit class)


## A finer point: `in_projection_period` (narrow) is NOT `rhna_credit_cycle` (wide)

Both windows **start** at 2022-06-30 - which makes them easy to confuse. But the projection period is a **narrow bridge** that *ends* 2023-01-30, while the 6th-cycle **credit** window has **no upper cap** (it runs the whole cycle). A 2024 first-BP is *not* in the projection period, yet it absolutely earns **6th-cycle credit**. Same start, different windows.

In [7]:
from datetime import date
for d in [date(2022, 9, 8), date(2024, 5, 1)]:
    print(f'  {d}:  is_projection_period={is_projection_period(d)!s:5}  rhna_credit_cycle={rhna_credit_cycle(d)}')
print('  -> same 2022-06-30 start; the 2024 date left the narrow bridge but still earns 6th credit.')

  2022-09-08:  is_projection_period=True   rhna_credit_cycle=6th
  2024-05-01:  is_projection_period=False  rhna_credit_cycle=6th
  -> same 2022-06-30 start; the 2024 date left the narrow bridge but still earns 6th credit.


## Which rung needs which

For the **minimal APR (Table A2, completions by year)** you only need **reporting_year**. `calendar_cycle` and `rhna_credit_cycle` feed **Table B / cumulative RHNA progress** - the advanced (and coverage-limited) rung. Keeping them as *separate* fields is what lets each table read the right one.

## Checkpoint

In [8]:
# 1) reporting_year distribution is sane (the 2018-2025 shape) and sums to all completions
by_year = Counter(reporting_year(b) for b in completions.values())
assert sum(by_year.values()) == 951, sum(by_year.values())
assert set(by_year) <= set(range(2018, 2027)) and by_year[2023] > by_year[2018]   # sane shape
# 2) the non-conflation, proven on one real building: calendar_cycle != rhna_credit_cycle
k = normalize_address('1951 Shattuck Ave'); b = spine[(k.number, k.street, k.stype)]
fbp = min(b['issue'])[0]
assert cycle_for_date(fbp) == '5th' and rhna_credit_cycle(fbp) == '6th' and is_projection_period(fbp)

print('CHECKPOINT PASS')
print(f'  reporting_year sums to {sum(by_year.values())} (== completions); sane 2018-2025 shape')
print('  non-conflation proven: 1951 Shattuck first-BP 2022-09-08 -> calendar 5th BUT credit 6th')

CHECKPOINT PASS
  reporting_year sums to 951 (== completions); sane 2018-2025 shape
  non-conflation proven: 1951 Shattuck first-BP 2022-09-08 -> calendar 5th BUT credit 6th


**JN5 done.** Each completion is tagged with the right *three* date-concepts, kept distinct. **Next - JN6 (the capstone):** generate the HCD-form APR, score it against the city's submitted numbers, and decompose the residual - where the 2352 Shattuck collapse you have tracked since JN3 finally gets rediscovered and named.